In [1]:
import import_ipynb
import numpy as np
import sys
from itertools import combinations, combinations_with_replacement

from functions import *

In [2]:
"""Create a grid of momenta"""

momentum_grid = [[px, py, pz] for pz in range(-5, 6) for py in range(-5, 6) for px in range(-5, 6)\
                 if (pz==0) and (px>0) and ((np.sqrt(px**2+py**2+pz**2)<=5))]
momentum_grid = np.array(momentum_grid)

In [3]:
"""Convert the momenta and kinetic energy grids to dictionaries"""

label_to_mom_dict = {i: item for i, item in enumerate(momentum_grid)}
#mom_to_label_dict = {tuple(key): item for item, key in label_to_mom_dict.items()}
label_to_kin_dict = {key: np.sqrt(val@val) for key, val in label_to_mom_dict.items()}

In [4]:
"""Create array of allowed momenta for 2->2 scattering"""

# create list of momentum pairs where p1 not distinguishable from p2 and p3 not distinguishable from p4
# remove any p1=p3
all_mom_combinations=list(combinations_with_replacement(list(combinations_with_replacement([*range(35)], 2)), 2))
all_mom_combinations = [[item[0][0], item[0][1], item[1][0], item[1][1]] for item in all_mom_combinations if item[0][0] != item[1][0]]

# select for pairs that conserve momentum and kinetic energy and g-function is nonzero
ok_mom_combinations=[item for item in all_mom_combinations\
 if (np.sum(np.abs(label_to_mom_dict[item[0]]+label_to_mom_dict[item[1]]-label_to_mom_dict[item[2]]-label_to_mom_dict[item[3]])) < 10**-7)\
 and (np.abs(label_to_kin_dict[item[0]]+label_to_kin_dict[item[1]]-label_to_kin_dict[item[2]]-label_to_kin_dict[item[3]]) < 10**-7)\
 and (np.abs(g_factor(label_to_mom_dict[item[0]], label_to_mom_dict[item[1]], label_to_mom_dict[item[2]], label_to_mom_dict[item[3]])) > 10**-7)]

# generate reverse pairings, e.g., (q1, q2, q3, q4) -> (q1, q2, q3, q4), (q3, q4, q1, q2)
mom_combinations = []
for item in ok_mom_combinations: 
    mom_combinations.append(item)
    mom_combinations.append([*item[2:], *item[:2]])
mom_combinations.sort()

mom_combinations=np.array(mom_combinations)

In [5]:
"""Create dictionary of allowed momentum transitions for 2->2 scattering"""

mom_combinations_dict={tuple(item[:2]): set(item[2:]) for item in mom_combinations}
for key, item in mom_combinations_dict.items():
    mom_combinations_dict[key] = mom_combinations_dict[key].union(set(item))

In [6]:
"""Create a list of all states accessed, starting with the initial state"""

#accessed_states = [list(initial_state)]

'Create a list of all states accessed, starting with the initial state'

In [7]:
"""Define functions that take a state as input and return its kinetic energy and momentum"""

def get_kin(state):
    running_kin = 0
    for i in state:
        running_kin += label_to_kin_dict[i]
    return running_kin

def get_mom(state):
    running_mom = np.zeros(3)
    for i in state:
        running_mom += np.array(label_to_mom_dict[i])
    return running_mom

In [28]:
"""See what states are accessed via 2->2 scatterings; run until the length of accessed_states is unchanged"""

def generate_states(state):
    if (len(state) > 1) and (type(state[0]) is list):
        accessed_states = state
    elif type(state[0]) is not list:
        accessed_states = [list(state)]
    num_states = len(accessed_states)
    # get the kinetic energy and momentum
    kin = get_kin(accessed_states[0])
    mom = get_mom(accessed_states[0])
    for state in accessed_states.copy():
        # iterate over all pairs of particles and see if there's an allowed transition
        for item in combinations_with_replacement(state, 2):
            try:
                # p1, p2 are the pair to try; p3, p4 are the state transitioned to
                p3, p4 = tuple(mom_combinations_dict[item])
                p1, p2 = item
                # create a dictionary to populate the new state
                subs_dict = {item: item for item in state}
                subs_dict[p1] = p3
                subs_dict[p2] = p4
                new_state = [subs_dict[item] for item in state]
                new_state.sort()
                # if the new state isn't already in accessed_states and kinetic energy and momentum are conserved, add it to accessed_states
                if (new_state not in accessed_states) and\
                (np.round(get_kin(new_state), 7)==np.round(kin, 7)) and (np.round(get_mom(new_state), 7)==np.round(mom, 7)).all():
                    accessed_states.append(sorted(new_state))
            except KeyError:
                pass
    # see if we've hit every state; if not, do again
    if num_states != len(accessed_states):
        generate_states(accessed_states)
    return accessed_states

In [9]:
# """See what states are accessed via 2->2 scatterings; run until the length of accessed_states is unchanged"""

# def generate_states():
#     num_states = len(accessed_states)
#     # get the kinetic energy and momentum
#     kin = get_kin(accessed_states[0])
#     mom = get_mom(accessed_states[0])
#     for state in accessed_states.copy():
#         # iterate over all pairs of particles and see if there's an allowed transition
#         for item in combinations_with_replacement(state, 2):
#             try:
#                 # p1, p2 are the pair to try; p3, p4 are the state transitioned to
#                 p3, p4 = tuple(mom_combinations_dict[item])
#                 p1, p2 = item
#                 # create a dictionary to populate the new state
#                 subs_dict = {item: item for item in state}
#                 subs_dict[p1] = p3
#                 subs_dict[p2] = p4
#                 new_state = [subs_dict[item] for item in state]
#                 new_state.sort()
#                 # if the new state isn't already in accessed_states and kinetic energy and momentum are conserved, add it to accessed_states
#                 if (new_state not in accessed_states) and\
#                 (np.round(get_kin(new_state), 7)==np.round(kin, 7)) and (np.round(get_mom(new_state), 7)==np.round(mom, 7)).all():
#                     accessed_states.append(new_state)
#             except KeyError:
#                 pass
#     # see if we've hit every state; if not, do again
#     if num_states != len(accessed_states):
#         generate_states()
#     return accessed_states

In [10]:
# generate_states()
# len(accessed_states)

In [25]:
# """Set the initial state"""

# initial_state = np.array([0,5,10,25,28,33])
# #initial_state = np.array([0,5,10,12,20,25,28,33])
# #initial_state = np.array([0,5,8,10,12,20,25,26,28,33])

In [26]:
# thing=generate_states(list(initial_state))

In [27]:
# len(thing)

12